# Dimensionality Reduction: Principal Component Analysis (PCA)

This notebook demonstrates the practical implementation of PCA using the `digits` dataset. We will cover:
1. **Preprocessing:** Why standardizing features is mandatory before PCA.
2. **Explained Variance (The Elbow Curve):** Determining how many components we actually need to retain 95% of the information.
3. **2D Projection:** Visualizing 64-dimensional data on a 2D plot.
4. **Interpreting Components:** Understanding what the components actually represent.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

sns.set_theme(style="whitegrid")

### 1. Load the Data
We are using the `load_digits` dataset. Each data point is an 8x8 image of a handwritten digit. 
Therefore, flattened out, **each image has 64 features (pixels)**.

In [ ]:
# Load dataset
digits = load_digits()
X = digits.data
y = digits.target

print(f"Data Shape: {X.shape} -> (1797 samples, 64 dimensions/features)")
print(f"Target Labels: {np.unique(y)} -> (Digits 0 through 9)")

# Let's visualize what one of these 64-dimensional rows actually looks like as an image
plt.figure(figsize=(3, 3))
plt.imshow(X[0].reshape(8, 8), cmap='gray_r')
plt.title(f"Label: {y[0]}")
plt.axis('off')
plt.show()

### 2. Preprocessing: The Scaling Imperative
PCA is a variance-maximizing algorithm. If one feature is measured in thousands and another in decimals, PCA will completely ignore the decimals and focus only on the massive numbers. **We must scale the data so all features have a mean of 0 and a variance of 1.**

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Mean of first feature before scaling:", X[:, 0].mean())
print("Mean of first feature after scaling:", np.round(X_scaled[:, 0].mean(), 5))

### 3. Finding the Optimal $k$: The Explained Variance Curve
How many components do we need to accurately represent these images? We fit PCA without restricting the components to see how much variance each additional component explains.

In [ ]:
# Fit PCA on all 64 dimensions
pca_full = PCA().fit(X_scaled)

# Calculate Cumulative Variance
cumulative_variance = np.cumsum(pca_full.explained_variance_ratio_)

# Plot the Elbow Curve
plt.figure(figsize=(10, 6))
plt.plot(range(1, 65), cumulative_variance, marker='.', linestyle='-', color='teal')

# Draw a line at 95% variance retention
plt.axhline(y=0.95, color='r', linestyle='--', label='95% Explained Variance')

# Find the exact number of components needed for 95%
k_95 = np.argmax(cumulative_variance >= 0.95) + 1
plt.axvline(x=k_95, color='r', linestyle=':')

plt.title("PCA: Cumulative Explained Variance vs. Number of Components", fontsize=14)
plt.xlabel("Number of Principal Components")
plt.ylabel("Cumulative Variance Explained")
plt.legend()
plt.show()

print(f"Insight: To retain 95% of the information, we only need {k_95} components out of 64.")
print("We have successfully compressed the dataset by about 37% with almost zero data loss!")

### 4. Dimensionality Reduction for Visualization (2D Projection)
Let's be extreme. Let's crush all 64 dimensions down to just **2 dimensions** so we can plot it on an X/Y graph. How well do the clusters of numbers separate?

In [ ]:
# 1. Extract exactly 2 components
pca_2d = PCA(n_components=2)
X_pca_2d = pca_2d.fit_transform(X_scaled)

print(f"New Data Shape: {X_pca_2d.shape}")
print(f"Variance explained by just 2 components: {np.sum(pca_2d.explained_variance_ratio_) * 100:.2f}%")

# 2. Visualize
plt.figure(figsize=(12, 8))
scatter = plt.scatter(X_pca_2d[:, 0], X_pca_2d[:, 1], c=y, cmap='tab10', alpha=0.7, edgecolor='k')

# Add a legend
legend = plt.legend(*scatter.legend_elements(), title="Digits", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.title("2D PCA Projection of 64-Dimensional Handwritten Digits", fontsize=16)
plt.xlabel(f"Principal Component 1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}%)")
plt.ylabel(f"Principal Component 2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}%)")
plt.tight_layout()
plt.show()

print("--- Diagnostic Insight ---")
print("Even with only ~21% of the total variance, PCA managed to group similar numbers together! ")
print("Look at the light blue (0) and orange (1)—they are distinctly separated. However, more complex digits blur together in the center. ")
print("To get perfect separation, we would need either more components (like the 40 we calculated above) or a non-linear algorithm like t-SNE.")